In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
import csv
from utils.babel import pybel, prepare_receptor, assign_charges, babel_diagnostics
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm


In [ ]:
INPUT = require_stage("05_pubchem") / "pubchem_unmatched_ligands.csv"
RECEPTOR = RAW / "3FXI.pdb"
SEED = 42

In [ ]:
with stage_run("06_preparation", [INPUT, RECEPTOR], {"seed": SEED}) as output_dir:
    ligands_dir = output_dir / "ligands"
    ligands_dir.mkdir()
    receptor = prepare_receptor(RECEPTOR)
    with babel_diagnostics():
        receptor.write("pdbqt", str(output_dir / "receptor.pdbqt"), overwrite=True, opt={"r": None, "c": None})
    with INPUT.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        columns = list(reader.fieldnames or [])
        if not {"Name", "SMILES"} <= set(columns):
            raise ValueError("Missing Name or SMILES column")
        rows = list(reader)
    with (output_dir / "preparation_results.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns + ["LigandID", "PreparationStatus", "PreparationError"])
        writer.writeheader()
        for index, row in enumerate(tqdm(rows, desc="Preparing ligands"), start=1):
            ligand_id = f"ligand_{index:06d}"
            status, error = "OK", ""
            try:
                mol = Chem.MolFromSmiles(row.get("CanonicalSMILES") or row["SMILES"])
                if mol is None:
                    raise ValueError("Invalid SMILES")
                mol = Chem.AddHs(mol)
                params = AllChem.ETKDGv3()
                params.randomSeed = SEED
                if AllChem.EmbedMolecule(mol, params) != 0:
                    params.useRandomCoords = True
                    if AllChem.EmbedMolecule(mol, params) != 0:
                        raise RuntimeError("3D embedding failed")
                if not AllChem.MMFFHasAllMoleculeParams(mol):
                    raise RuntimeError("MMFF parameters unavailable")
                if AllChem.MMFFOptimizeMolecule(mol, maxIters=1000) != 0:
                    raise RuntimeError("MMFF optimization did not converge")
                prepared = pybel.readstring("mol", Chem.MolToMolBlock(mol))
                with babel_diagnostics():
                    assign_charges(prepared.OBMol)
                prepared.write("pdbqt", str(ligands_dir / f"{ligand_id}.pdbqt"), overwrite=True)
            except Exception as exc:
                status, error = "ERROR", str(exc)
                logger.exception("Preparation failed: {} ({})", ligand_id, row["Name"])
            writer.writerow(dict(row, LigandID=ligand_id, PreparationStatus=status, PreparationError=error))
    logger.info("Preparation records: {}", len(rows))
